In [173]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import folium
from folium.plugins import pattern
from folium.elements import MacroElement
import branca.colormap as cm
from branca.element import Element
# import streamlit as st
# from streamlit_folium import st_folium

gdf = gpd.read_file("data/map2026.geojson")

# Check for EPSG 4326 for folium compatibility
print(gdf.crs)
gdf.head(8)

EPSG:4326


,District No.,District,Harris New,Trump New,Margin New,Margin %,Margin Shift,Margin Shift %,Partisan Index,Harris 24,Trump 24,Margin 24,Margin New %,Targeted,geometry
0,1,AL01,0.314327,0.673261,-0.358934,-55.32%,0.194288,19.43%,-0.185673,0.219426,0.772648,-0.553222,-35.89%,False,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.143004,8.23%,-0.225346,-22.53%,-0.077173,0.536501,0.454159,0.082342,-14.30%,True,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.465175,-46.69%,0.001756,0.18%,-0.238118,0.262503,0.729434,-0.466931,-46.52%,False,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.663093,-66.43%,0.001169,0.12%,-0.336624,0.164162,0.828424,-0.664262,-66.31%,False,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.289816,-29.11%,0.001269,0.13%,-0.153897,0.347914,0.638999,-0.291085,-28.98%,False,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.341998,-38.55%,0.043526,4.35%,-0.179472,0.301863,0.687387,-0.385524,-34.20%,False,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.177735,23.36%,-0.055874,-5.59%,0.082883,0.612241,0.378632,0.233609,17.77%,False,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,CA01,0.544906,0.422732,0.122174,-24.95%,0.371672,37.17%,0.044906,0.361250,0.610747,-0.249498,12.22%,True,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308..."


In [174]:
# Test data visualization with matplotlib
# Plot 2024 Pres. margin and shift under new maps 
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# gdf.plot(
#     column='Margin New', 
#     cmap='RdBu',
#     vmin=-0.5,
#     vmax=0.5,
#     legend=True,              
#     legend_kwds={'label': 'Harris-Trump % Margin',
#                  'orientation': 'horizontal',
#                  'shrink': 0.7,
#                  'pad': 0.05},
#     edgecolor='black',        
#     ax=ax1
# )

# ax1.axis('off')
# ax1.set_title(
#     "2024 Pres. Margin Under New Boundaries", 
#     fontsize=16, 
#     fontweight='bold'
# )

# Change legend markers to percentage
# fig.axes[2].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# gdf.plot(
#     column='Margin Shift',      
#     cmap='RdBu',
#     vmin=-0.5,
#     vmax=0.5,
#     legend=True,              
#     legend_kwds={'label': 'Harris-Trump % Margin',
#                  'orientation': 'horizontal',
#                  'shrink': 0.7,
#                  'pad': 0.05},
#     edgecolor='black',        
#     ax=ax2
# )

# ax2.axis('off')
# ax2.set_title(
#     "2024 Pres. Margin Shift from Old Boundaries",
#     fontsize=16,
#     fontweight='bold'
# ) 

# fig.axes[3].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# plt.tight_layout()
# plt.show()

In [175]:
# Create new columns that shows partisan lean (e.g. D+7.89) for visualization tooltips
def partisan_text(val):
    if val is None or str(val).strip() in ['', 'nan', 'None']:
        return "EVEN"
        
    try:
        num = val * 100
        
        if num < 0:
            return f"Trump+{abs(num):.2f}"
            
        elif num > 0:
            return f"Harris+{num:.2f}"
            
        return "EVEN"
    except ValueError:
        return str(val)

gdf['Margin Partisan'] = gdf['Margin 24'].apply(partisan_text)
gdf['Margin New Partisan'] = gdf['Margin New'].apply(partisan_text)
gdf['Margin Shift Partisan'] = gdf['Margin Shift'].apply(partisan_text)

gdf.head(8)

,District No.,District,Harris New,Trump New,Margin New,Margin %,Margin Shift,Margin Shift %,Partisan Index,Harris 24,Trump 24,Margin 24,Margin New %,Targeted,geometry,Margin Partisan,Margin New Partisan,Margin Shift Partisan
0,1,AL01,0.314327,0.673261,-0.358934,-55.32%,0.194288,19.43%,-0.185673,0.219426,0.772648,-0.553222,-35.89%,False,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",Trump+55.32,Trump+35.89,Harris+19.43
1,2,AL02,0.422827,0.565831,-0.143004,8.23%,-0.225346,-22.53%,-0.077173,0.536501,0.454159,0.082342,-14.30%,True,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",Harris+8.23,Trump+14.30,Trump+22.53
2,3,AL03,0.261882,0.727057,-0.465175,-46.69%,0.001756,0.18%,-0.238118,0.262503,0.729434,-0.466931,-46.52%,False,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",Trump+46.69,Trump+46.52,Harris+0.18
3,4,AL04,0.163376,0.826469,-0.663093,-66.43%,0.001169,0.12%,-0.336624,0.164162,0.828424,-0.664262,-66.31%,False,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",Trump+66.43,Trump+66.31,Harris+0.12
4,5,AL05,0.346103,0.635919,-0.289816,-29.11%,0.001269,0.13%,-0.153897,0.347914,0.638999,-0.291085,-28.98%,False,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",Trump+29.11,Trump+28.98,Harris+0.13
5,6,AL06,0.320528,0.662526,-0.341998,-38.55%,0.043526,4.35%,-0.179472,0.301863,0.687387,-0.385524,-34.20%,False,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",Trump+38.55,Trump+34.20,Harris+4.35
6,7,AL07,0.582883,0.405147,0.177735,23.36%,-0.055874,-5.59%,0.082883,0.612241,0.378632,0.233609,17.77%,False,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",Harris+23.36,Harris+17.77,Trump+5.59
7,1,CA01,0.544906,0.422732,0.122174,-24.95%,0.371672,37.17%,0.044906,0.361250,0.610747,-0.249498,12.22%,True,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308...",Trump+24.95,Harris+12.22,Harris+37.17


In [176]:
# Finding map center
centroid = gdf.unary_union.centroid
center = [centroid.y, centroid.x]

# Initiate folium map. Tilelayer is initiated separately to prevent it as a toggle option
m = folium.Map(
    location=center, 
    zoom_start=5, 
    tiles=None,
    min_zoom=5,
    max_zoom=8
)

m.options['maxZoom'] = 8

folium.TileLayer(
    tiles="cartodbpositron", 
    name="Base Map", 
    control=False,
    min_zoom=5,
    max_zoom=8
).add_to(m)

In [177]:
# Define styling for targeted districts
hatch_pattern = pattern.StripePattern(
    angle=-45,
    color='black',
    space_color='transparent',
    weight=3,
    space_weight=5
).add_to(m)

# Define function for hatch styling to avoid browser ordering issues when rendering
def style_hatch(feature):
    targeted = feature['properties'].get('Targeted', False)
    if targeted:
        return {'fillPattern': hatch_pattern, 'fillOpacity': 0.6, 'color': 'transparent'}
    return {'fillColor': 'transparent', 'color': 'transparent'}

In [178]:
# Color scheme for partisan lean of new districts
def style_left(feature):
    margin = feature['properties'].get('Margin New', 0)
    
    if margin >= 0.30:     color = '#084594'  
    elif margin >= 0.15:   color = 'steelblue'  
    elif margin > 0:       color = 'lightblue' 
    elif margin >= -0.15:  color = 'lightcoral' 
    elif margin >= -0.30:  color = 'indianred'  
    else:                  color = 'firebrick' 
        
    style_dict = {
        'fillColor': color,
        'color': 'black',   
        'weight': 1,
        'fillOpacity': 0.8
    }
        
    return style_dict

# Color scheme for partisan shift for new boundaries
def style_right(feature):
    margin = feature['properties'].get('Margin Shift', 0)
    
    if margin >= 0.30:     color = '#084594'  
    elif margin >= 0.15:   color = 'steelblue'  
    elif margin > 0:       color = 'lightblue' 
    elif margin >= -0.15:  color = 'lightcoral' 
    elif margin >= -0.30:  color = 'indianred'  
    else:                  color = 'firebrick' 
        
    style_dict = {
        'fillColor': color,
        'color': 'black',   
        'weight': 1,
        'fillOpacity': 0.8
    }
        
    return style_dict

In [179]:
# Tooltip styling
tooltip_left = folium.GeoJsonTooltip(
    fields=['District No.', 'Margin New Partisan'], 
    aliases=['New District:', '2024 Presidential Margin:'],
    style=(
        "background-color: rgba(255, 255, 255, 0.95); "  
        "color: #1a1a1a; "                               
        "font-size: 12px; "                              
        "font-weight: bold; "
        "border: 2px solid #222222; "                    
        "box-shadow: 3px 3px 10px rgba(0,0,0,0.25);"     
    ),
    localize=True
)

tooltip_right = folium.GeoJsonTooltip(
    fields=['District No.', 'Margin Shift Partisan', 'Margin Partisan'], 
    aliases=['New District:', "Shift from Old District's Margin:", "Old District's 2024 Margin:"],
    style=(
        "background-color: rgba(255, 255, 255, 0.95); "
        "color: #1a1a1a; "
        "font-size: 12px; "
        "font-weight: bold; "
        "border: 2px solid #222222; "
        "border-radius: 4px; "
        "box-shadow: 3px 3px 10px rgba(0,0,0,0.25);"
    ),
    localize=True
)

# Make the 3rd field in the shift map tooltip normal font to indicate less importance
normal_font = """
<style>
.leaflet-tooltip tr:nth-child(3) th,
.leaflet-tooltip tr:nth-child(3) td {
    font-weight: normal !important;
}
</style>
"""

# Append the above font rule to the map head layout
m.get_root().header.add_child(folium.Element(normal_font))

In [ ]:
# Create a feature group for each map to be toggled
group_lean = folium.FeatureGroup(name="Margin LEAN of 2026 Districts", overlay=False, show=True).add_to(m)
group_shift = folium.FeatureGroup(name="Margin SHIFT from 2024 Districts", overlay=False, show=False).add_to(m)

# Bind data to their respective feature groups
folium.GeoJson(
    gdf,
    style_function=style_left,
).add_to(group_lean)

folium.GeoJson(
    gdf,
    style_function=style_hatch,
    tooltip=tooltip_left
).add_to(group_lean)

folium.GeoJson(
    gdf,
    style_function=style_right,
).add_to(group_shift)

folium.GeoJson(
    gdf,
    style_function=style_hatch,
    tooltip=tooltip_right
).add_to(group_shift)

In [ ]:
# Add a custom city label layer on top of the maps
pane_labels = folium.map.CustomPane("labels_top", z_index=450).add_to(m)

folium.TileLayer(
    tiles="cartodbpositrononlylabels", 
    pane="labels_top", 
    name="City Labels",
    control=False,
    min_zoom=5,
    max_zoom=8
).add_to(m)

In [ ]:
# # 1. Custom CSS and HTML structure for the control panel buttons
# custom_button_html = """
# <div id="map-toggle-panel">
#     <button id="btn-lean" class="toggle-btn active-btn">Margin LEAN of 2026 Districts</button>
#     <button id="btn-shift" class="toggle-btn">Margin SHIFT from 2024 Districts</button>
# </div>

# <style>
#     #map-toggle-panel {
#         position: absolute;
#         top: 20px;
#         left: 20px;
#         z-index: 1000;
#         background: white;
#         padding: 6px;
#         border-radius: 6px;
#         border: 2px solid #222222;
#         box-shadow: 3px 3px 10px rgba(0,0,0,0.2);
#         display: flex;
#         gap: 6px;
#     }
#     .toggle-btn {
#         background-color: #f1f1f1;
#         color: #333;
#         border: 1px solid #ccc;
#         padding: 8px 14px;
#         font-size: 14px;
#         font-weight: bold;
#         cursor: pointer;
#         border-radius: 4px;
#         transition: all 0.2s ease;
#     }
#     .toggle-btn:hover {
#         background-color: #e0e0e0;
#     }
#     .active-btn {
#         background-color: #222222 !important;
#         color: white !important;
#         border-color: #222222 !important;
#     }
# </style>
# """
# m.get_root().html.add_child(folium.Element(custom_button_html))


# # 2. Custom JavaScript to tie the buttons to Leaflet's layer manipulation system
# # This maps our button clicks to the underlying Leaflet unique layer IDs automatically
# layer_switching_js = """
# <script>
# document.addEventListener("DOMContentLoaded", function() {
#     var checkMapInterval = setInterval(function() {
#         // Wait until Leaflet map and our layers are fully initiated
#         if (typeof folium_map_html !== 'undefined' || typeof m !== 'undefined') {
#             clearInterval(checkMapInterval);
            
#             // Resolve the folium map object variable dynamically
#             var mapObj = (typeof m !== 'undefined') ? m : window[Object.keys(window).find(k => k.startsWith('map_'))];
            
#             var leanLayer, shiftLayer;
            
#             // Find our feature group layers inside Leaflet
#             mapObj.eachLayer(function(layer) {
#                 if (layer.options && layer.options.name === "Margin LEAN") {
#                     leanLayer = layer;
#                 }
#                 if (layer.options && layer.options.name === "Margin SHIFT") {
#                     shiftLayer = layer;
#                 }
#             });

#             var btnLean = document.getElementById('btn-lean');
#             var btnShift = document.getElementById('btn-shift');

#             btnLean.addEventListener('click', function() {
#                 if (!mapObj.hasLayer(leanLayer)) {
#                     mapObj.addLayer(leanLayer);
#                     mapObj.removeLayer(shiftLayer);
#                     btnLean.classList.add('active-btn');
#                     btnShift.classList.remove('active-btn');
#                 }
#             });

#             btnShift.addEventListener('click', function() {
#                 if (!mapObj.hasLayer(shiftLayer)) {
#                     mapObj.addLayer(shiftLayer);
#                     mapObj.removeLayer(leanLayer);
#                     btnShift.classList.add('active-btn');
#                     btnLean.classList.remove('active-btn');
#                 }
#             });
#         }
#     }, 100);
# });
# </script>
# """
# m.get_root().html.add_child(folium.Element(layer_switching_js))

In [ ]:
# Initiate a legend
bounds = [-0.45, -0.30, -0.15, 0, 0.15, 0.30, 0.45]
colors = ['firebrick', 'indianred', 'lightcoral', 'lightblue', 'steelblue', '#084594']

legend = cm.StepColormap(
    colors=colors,
    index=bounds,
    vmin=min(bounds),
    vmax=max(bounds),
    caption="Harris-Trump Margin / Margin Shift (Striped = Redistricted to Flip)"
)

legend.width = 650
legend.height = 45

# Custom CSS for legend styling
legend_css = """
<style>
    /* Container */
    .legend {
        position: fixed !important;
        bottom: 20px !important;
        left: 20px !important;
        z-index: 9999 !important; /* Ensures it stays on top of district polygons */
        background-color: rgba(255, 255, 255, 0.95) !important;        
        border: 2px solid #222222 !important;       
        border-radius: 6px !important;              
        padding: 12px !important;
        box-shadow: 3px 3px 12px rgba(0,0,0,0.15);
    }
    
    /* Element visibility regardless of positioning */
    .legend svg {
        height: 55px !important; 
        overflow: visible !important;
    }
    
    /* Title */
    .legend .caption {
        color: black !important;                  
        font-size: 14px !important;
        font-weight: bold;
        transform: translateY(10px);
    }
    
    /* Ticks */
    .legend text {
        fill: black !important;                   
        font-size: 11px !important;
    }
</style>
"""
m.get_root().header.add_child(folium.Element(legend_css))

# javascript to convert ticks from decimals to percentages
percentage_formatter_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    // Array matching your 7 explicit tick bounds sequentially
    var customLabels = ['-45%', '-30%', '-15%', '0%', '+15%', '+30%', '+45%'];
    
    var ticks = document.querySelectorAll('div.legend g.tick text');
    
    // Explicitly overwrite the text of each generated tick position
    for (var i = 0; i < ticks.length; i++) {
        if (i < customLabels.length) {
            ticks[i].textContent = customLabels[i];
        }
    }
});
</script>
"""
m.get_root().html.add_child(folium.Element(percentage_formatter_js))

# The semicolon prevents automatic visual rendering; alternatively, assign this to a dummy variable
m.add_child(legend);

In [ ]:
# JS and CSS injection for header container, including title and toggle box
unified_header_html = """
<div id="header-wrapper">
    <div class="title-card">
        <h1 style="margin: 0; font-size: 20px; font-weight: bold;">
            Mid-Decade Redistricting Visualized
        </h1>
        <p style="margin: 3px 0 0 0; font-size: 12px; font-weight: bold;">
            Partisan Lean and Shift of New Districts based on the 2024 Presidential Margin
        </p>
    </div>
    
    <!-- Empty anchor destination where our CSS will inject and position the native toggle box -->
    <div id="toggle-anchor-zone"></div>
</div>

<style>
    /* Master Flex wrapper to structurally stack components without fixed overlapping gaps */
    #header-wrapper {
        position: fixed;
        top: 15px;
        left: 50%;
        transform: translateX(-50%);
        z-index: 9999;
        display: flex;
        flex-direction: column;
        align-items: center;
        gap: 12px;               
        width: 90%;             
        max-width: 650px;
        pointer-events: none;    /* Allows dragging map through empty gap spaces */
    }
    
    .title-card {
        pointer-events: auto;    /* Enables interaction over text fields */
        /* The 4th value is opacity */
        background: linear-gradient(
            to right, 
            rgba(240, 128, 128, 0.85), 
            rgba(245, 245, 245, 0.85), 
            rgba(135, 206, 250, 0.85)  
        );
        color: #1a1a1a; 
        border: 2px solid #222222;
        border-radius: 6px;
        padding: 10px 20px;
        box-shadow: 3px 3px 12px rgba(0,0,0,0.15);
        text-align: center;
        font-family: 'Helvetica Neue', Arial, sans-serif;
        width: 80%;
        box-sizing: border-box;
    }
    
    .leaflet-top.leaflet-left {
        display: none !important; 
    }
    
    .leaflet-control-layers {
        pointer-events: auto;
        position: static !important; /* Strips original fixed overlay behaviors */
        margin: 0 auto !important;
        background-color: rgba(255, 255, 255, 0.95) !important;
        border: 2px solid #222222 !important;
        border-radius: 6px !important;
        box-shadow: 3px 3px 12px rgba(0,0,0,0.15) !important;
        padding: 6px 12px !important;
        font-family: 'Helvetica Neue', Arial, sans-serif !important;
        display: inline-block !important;
    }
    
    /* Change toggles from vertical to horizontal list */
    .leaflet-control-layers-list,
    .leaflet-control-layers-base {
        display: flex !important;
        flex-direction: row !important;
        align-items: center !important;
        justify-content: center !important;
        gap: 16px !important;
        flex-wrap: wrap !important; /* Wraps items cleanly on extra small screens */
    }
    
    .leaflet-control-layers-base label {
        display: flex !important;
        align-items: center !important;
        gap: 6px !important;
        margin: 0 !important;
        font-size: 13px !important;
        font-weight: bold !important;
        color: #1a1a1a !important;
        cursor: pointer !important;
        white-space: nowrap !important; /* Keeps individual labels on a single line */
    }
    
    .leaflet-control-layers-base input[type="radio"] {
        margin: 0 !important;
        cursor: pointer !important;
    }
</style>
"""
m.get_root().html.add_child(folium.Element(unified_header_html))

# Moves the toggle box from its native left-side positioning to within our header container
append_control_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    // Checks for element loads every 100ms
    var checkControlInterval = setInterval(function() {
        var nativeControl = document.querySelector('.leaflet-control-layers');
        var targetAnchor = document.getElementById('toggle-anchor-zone');
        
        if (nativeControl && targetAnchor) {
            // If elements have loaded, stop the timer check and reposition the toggle box
            clearInterval(checkControlInterval);
            targetAnchor.appendChild(nativeControl);
        }
    }, 100);
});
</script>
"""
m.get_root().html.add_child(folium.Element(append_control_js))

# Initiate the toggle box
folium.LayerControl(position='topleft', collapsed=False).add_to(m)

m.save("index.html")
m